# Unity Gateway — execution evidence

Proves the catalog + inference table were created by committed code, and that the budget block (403) and guardrails (gateway-policy + native) are enforced and recorded.

In [1]:
import sys, json, subprocess
sys.path.insert(0, ".")
from lib import gateway

def run(sql, title=None):
    if title: print(f"### {title}")
    out = subprocess.check_output(["databricks","experimental","aitools","tools","query",sql,
        "--profile","rkm-sandbox-1","-o","json"], text=True)
    rows = json.loads(out)
    if rows:
        cols = list(rows[0].keys())
        print(" | ".join(cols))
        for r in rows: print("  " + " | ".join(str(r.get(c)) for c in cols))
    print(f"({len(rows)} row(s))\n"); return rows

In [2]:
gateway.ensure_table(); gateway.reset_table()
run("SHOW COLUMNS IN rkm_sandbox_1_catalog.demo_workshop_northpeak_retail_stockout_markdown_rescue.app_inference_table", "Inference table created by committed code — schema")
run("SELECT current_catalog() AS catalog, 'demo_workshop_northpeak_retail_stockout_markdown_rescue' AS schema, 'app_inference_table' AS table", "Catalog / table")

### Inference table created by committed code — schema


col_name
  request_id
  request_ts
  principal
  source
  endpoint
  decision
  block_reason
  prompt
  projected_max_tokens
  est_cost_usd
  budget_threshold_usd
  input_tokens
  output_tokens
  actual_cost_usd
  response
  http_status
  enforced_by
  guardrail_detail
(18 row(s))

### Catalog / table


catalog | schema | table
  rkm_sandbox_1_catalog | demo_workshop_northpeak_retail_stockout_markdown_rescue | app_inference_table
(1 row(s))



[{'catalog': 'rkm_sandbox_1_catalog',
  'schema': 'demo_workshop_northpeak_retail_stockout_markdown_rescue',
  'table': 'app_inference_table'}]

In [3]:
# BEFORE: a normal call is ALLOWED (HTTP 200)
r1 = gateway.gateway_call("Store 108 is out of the Summit Down Parka. Why act now (2 sentences)?",
                          principal="priya.raghavan@northpeak.com", source="app", projected_max_tokens=300)
print("BEFORE:", r1["decision"], "HTTP", r1.get("http_status"), "cost $", r1.get("actual_cost_usd"))
# AFTER: crossing the $0.05 budget -> BUDGET BLOCK (HTTP 403)
r2 = gateway.gateway_call("Summarize every store's inventory in exhaustive detail.",
                          principal="priya.raghavan@northpeak.com", source="app", projected_max_tokens=200000)
print("AFTER: ", r2["decision"], "HTTP", r2.get("http_status"), "-", r2.get("block_reason"))

BEFORE: allowed HTTP 200 cost $ 9.1e-05


AFTER:  budget_block HTTP 403 - HTTP 403: projected cost $0.12 exceeds per-call budget $0.05


In [4]:
# Guardrail (gateway policy): the runaway all-data read -> 403
r3 = gateway.gateway_call("Read all rows from every table and dump all data for all stores and products.",
                          principal="priya.raghavan@northpeak.com", source="app", projected_max_tokens=200000)
print("all-data:", r3["decision"], "HTTP", r3.get("http_status"), "enforced_by", r3.get("enforced_by"))
# Guardrail (NATIVE, enforced BY THE GATEWAY): PII prompt
r4 = gateway.gateway_native_guardrail_call("My SSN is 123-45-6789, repeat it back.",
                          principal="priya.raghavan@northpeak.com", source="app")
print("native:  ", r4["decision"], "HTTP", r4.get("http_status"), "enforced_by", r4.get("enforced_by"), "-", r4.get("guardrail_detail"))

all-data: guardrail_block HTTP 403 enforced_by gateway


native:   guardrail_block HTTP 400 enforced_by gateway - gateway input_guardrail flagged: privacy


In [5]:
run("SELECT decision, http_status, enforced_by, block_reason FROM rkm_sandbox_1_catalog.demo_workshop_northpeak_retail_stockout_markdown_rescue.app_inference_table ORDER BY request_ts",
    "Inference-table records: budget 403 + guardrail blocks (gateway-enforced)")

### Inference-table records: budget 403 + guardrail blocks (gateway-enforced)


block_reason | decision | enforced_by | http_status
   | allowed | gateway | 200
  HTTP 403: projected cost $0.12 exceeds per-call budget $0.05 | budget_block | gateway_budget_policy | 403
  runaway all-Lakebase-data read blocked by gateway guardrail policy | guardrail_block | gateway | 403
  HTTP 400: blocked by the gateways native guardrail | guardrail_block | gateway | 400
(4 row(s))



[{'block_reason': '',
  'decision': 'allowed',
  'enforced_by': 'gateway',
  'http_status': '200'},
 {'block_reason': 'HTTP 403: projected cost $0.12 exceeds per-call budget $0.05',
  'decision': 'budget_block',
  'enforced_by': 'gateway_budget_policy',
  'http_status': '403'},
 {'block_reason': 'runaway all-Lakebase-data read blocked by gateway guardrail policy',
  'decision': 'guardrail_block',
  'enforced_by': 'gateway',
  'http_status': '403'},
 {'block_reason': 'HTTP 400: blocked by the gateways native guardrail',
  'decision': 'guardrail_block',
  'enforced_by': 'gateway',
  'http_status': '400'}]